In [ ]:
import pandas as pd
import geopandas as gpd
import datetime

# LA County Wildfire Data Download (January 2025)

See ArcGIS Viewer of Data Set here: 
https://data.lacounty.gov/datasets/6241d8e277a541a2b3645947d991c35e/explore?location=34.270980%2C-118.419280%2C8.35


In [ ]:
ipaws_raw = gpd.read_file('https://stg-arcgisazurecdataprod.az.arcgis.com/exportfiles-3435-135684/IPAWS_Jan_2025_Fire_Alerts_1540429968819036046.gpkg?sv=2018-03-28&sr=b&sig=WeJg5Dqjn8fC2VB2fjLzg88D9rOnW0XIzQ7btCrGUT8%3D&se=2025-05-20T23%3A50%3A19Z&sp=r')
ipaws_raw

In [ ]:
ipaws = ipaws_raw.copy()

# Format Columns
ipaws['Effective'] = pd.to_datetime(ipaws['Effective'])
ipaws['Expires'] = pd.to_datetime(ipaws['Expires'])
ipaws['CreateDate'] = pd.to_datetime(ipaws['CreateDate'])
ipaws['Sent'] = pd.to_datetime(ipaws['Sent'])


In [ ]:
# Identify and remove empty columns & single value columns
ipaws = ipaws.drop([col for col in ipaws.columns if ipaws[col].isnull().all()], axis = 1)
ipaws = ipaws.loc[:, ipaws.nunique() > 1]

In [ ]:

# Filter to Dates of Interest
ipaws = ipaws[
    (
        (ipaws["Effective"] >= pd.Timestamp('2025-01-07', tz = 'America/Los_Angeles')) & 
        (ipaws["Effective"] >= pd.Timestamp('2025-01-10', tz = 'America/Los_Angeles'))
    ) | 
    (
        pd.isna(ipaws['Effective']) | 
        (ipaws["Expires"] >= pd.Timestamp('2025-01-07', tz = 'America/Los_Angeles')) & 
        (ipaws["Expires"] >= pd.Timestamp('2025-01-10', tz = 'America/Los_Angeles'))
    ) | 
    pd.isna(ipaws['Expires']) 
]
ipaws = ipaws[ipaws["Sent"] <= pd.Timestamp('2025-01-10', tz = 'America/Los_Angeles')]


In [ ]:

# Determine Type of Alert
ipaws['Type'] = ipaws.Headline.str.extract('(Alert|Order|Warning)')
ipaws.loc[54,'Type'] = 'Order'

# Fill in start date where "effective" is missing
ipaws['Start'] = ipaws['Effective'].fillna(ipaws['Sent'])
ipaws['End'] = ipaws['Expires']

ipaws.explore('Type')
